# V28: 3D rocket landing on MuJoCo (Colab)

First 3D policy. A 6-DOF vehicle with a gimballed engine learns to land inside a 0.5 m radius under wind, engine lag, sensor noise and randomized mass / thrust.

**Trained from scratch.** V27 (2D, 99.2% precise) cannot be loaded: the 3D vehicle has 16 observations and 4 controls instead of 9 and 2. The method is the same one that produced V27.

**Number to beat:** the PID baseline in `src/sim3d/baseline.py`, measured in Cell 4.

Before you run: **Runtime > Change runtime type > T4 GPU**. Run the cells top to bottom. If Colab disconnects during Cell 8, run Cells 1 to 3 again and then Cell 8: it resumes from the last checkpoint on Drive.

## Cell 1: GPU and Google Drive

In [ ]:
!nvidia-smi | head -15
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rocket_v28

## Cell 2: install
MuJoCo is a normal pip package. Colab already has PyTorch.

In [ ]:
!pip install -q mujoco "stable-baselines3>=2.4" "gymnasium>=1.0" tensorboard imageio imageio-ffmpeg
import mujoco, gymnasium, stable_baselines3, torch
print('mujoco', mujoco.__version__, '| gymnasium', gymnasium.__version__,
      '| sb3', stable_baselines3.__version__, '| torch', torch.__version__,
      '| cuda', torch.cuda.is_available())

## Cell 3: get the code
If the repo is private, use `https://<token>@github.com/...` with a fine-grained token that has read access to this repo.

In [ ]:
REPO = 'https://github.com/rajivagarwal221/Rocket-Land.git'
!rm -rf /content/rocket && git clone --depth 1 {REPO} /content/rocket
%cd /content/rocket
!ls src/sim3d src/train_v28_mujoco.py src/evaluate_3d.py src/watch_3d.py

## Cell 4: prove the simulator works (PID baseline, about 2 minutes)
A classical controller with perfect state knowledge, scored at the ship bar. Its PRECISE rate is the number V28 has to beat.

In [ ]:
!python src/evaluate_3d.py --baseline --episodes 300 --out /content/drive/MyDrive/rocket_v28/logs/pid_baseline.csv

## Cell 5: watch the baseline fly in 3D
Renders off-screen on the GPU and plays the video inline.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
!MUJOCO_GL=egl python src/watch_3d.py --baseline --episodes 2 --camera chase --video /content/baseline.mp4
from IPython.display import Video
Video('/content/baseline.mp4', embed=True, width=720)

## Cell 6: TensorBoard
Watch `eval/precise_rate`, `outcomes/*` and `curriculum/*`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/rocket_v28/logs/tb_v28

## Cell 7: wiring test (200k steps)
Checks parallel envs, curriculum, evaluation, checkpoints and resume on this machine before the long run. It writes to a throwaway folder so it cannot be mistaken for the real run.

Read the `fps` value in the log: `fps x 3600` is your steps per hour, which tells you how long Cell 8 will take.

In [ ]:
import os
os.environ.update(V28_OUT='/content/v28_wiring', V28_STEPS='200000', V28_NENVS='4', V28_FRESH='1')
!python src/train_v28_mujoco.py

## Cell 8: full training run (6M steps)
Checkpoints go to Drive every 100k steps, the replay buffer every 500k. **Re-running this cell resumes**; it never starts over unless you set `V28_FRESH='1'`.

The best checkpoint is chosen by PRECISE rate at full difficulty, printed as `[eval] ... <- new best`.

In [ ]:
import os
os.environ.update(V28_OUT='/content/drive/MyDrive/rocket_v28', V28_STEPS='6000000', V28_NENVS='4', V28_FRESH='0')
!python src/train_v28_mujoco.py

## Cell 9: audit the best checkpoint (1000 seeded episodes at the ship bar)

In [ ]:
!python src/evaluate_3d.py --model /content/drive/MyDrive/rocket_v28/models/master_v28_best.zip \
    --episodes 1000 --device cuda --out /content/drive/MyDrive/rocket_v28/logs/v28_best_audit.csv

## Cell 10: watch V28 land

In [ ]:
!MUJOCO_GL=egl python src/watch_3d.py --model /content/drive/MyDrive/rocket_v28/models/master_v28_best.zip \
    --episodes 3 --camera chase --video /content/v28.mp4
from IPython.display import Video
Video('/content/v28.mp4', embed=True, width=720)

## Cell 11: download the model and its manifest

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/rocket_v28/models/master_v28_best.zip')
files.download('/content/drive/MyDrive/rocket_v28/models/master_v28_best_manifest.json')